# 03 · Chunk the 10-Q

Stage 3 of the indexing pipeline (see `ARCHITECTURE.md`).

Cuts the parsed blocks into passages (chunks) that will be embedded and searched. Two strategies, compared side by side:

1. **Fixed-size**: `RecursiveCharacterTextSplitter`, ~1,000 characters with 150 overlap, breaking at paragraphs/sentences where it can
2. **Semantic**: `SemanticChunker`, which cuts where the meaning of neighbouring sentences shifts (uses the `all-MiniLM-L6-v2` embedding model), capped at 1,000 characters

Both strategies share the same **table chunks**: every table stays whole with its caption, and only long tables are split by rows (repeating their header rows).

⚠️ Memory: this notebook peaks at ~700 MB RAM (mostly the embedding model; one-time ~90 MB download). Close other notebook kernels first if RAM is tight.

Output: `data/chunks/fixed.jsonl` and `data/chunks/semantic.jsonl`.

In [ ]:
import json
import os
from itertools import groupby
from pathlib import Path

# Hugging Face warns that Windows can't use symlinks in its model cache; caching still works.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import matplotlib.pyplot as plt
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

PARSED_PATH = Path("../data/parsed/acre-20260630.json")
CHUNKS_DIR = Path("../data/chunks")
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
SOURCE = "acre-20260630.htm"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1000

blocks = json.loads(PARSED_PATH.read_text(encoding="utf-8"))
print(len(blocks), "blocks loaded")

## Prepare inputs

- **Text:** consecutive paragraphs in the same section are joined into one document per section, so the splitters can work across paragraph boundaries without ever mixing two sections.
- **Tables:** turned into chunks directly. A table longer than `TABLE_MAX` characters is split by rows, and each piece repeats the table's first two rows (the column headers).

In [ ]:
TABLE_MAX = 800


def section_of(block):
    return " · ".join(x for x in (block["part"], block["item"]) if x)


text_blocks = [b for b in blocks if b["type"] == "text"]
section_docs = [
    Document(
        page_content="\n\n".join(b["text"] for b in group),
        metadata={"source": SOURCE, "section": section, "type": "text"},
    )
    for section, group in groupby(text_blocks, key=section_of)
]


def table_chunks(block):
    lines = block["text"].splitlines()
    header = lines[:2]
    pieces, current = [], []
    for line in lines:
        if current and len("\n".join(current + [line])) > TABLE_MAX:
            pieces.append(current)
            current = list(header)
        current.append(line)
    pieces.append(current)

    meta = {"source": SOURCE, "section": section_of(block), "type": "table", "caption": block["caption"]}
    return [
        Document(
            page_content=f"Table: {block['caption']}{' (continued)' if i else ''}\n" + "\n".join(piece),
            metadata=dict(meta),
        )
        for i, piece in enumerate(pieces)
    ]


table_docs = [chunk for b in blocks if b["type"] == "table" for chunk in table_chunks(b)]

print(f"{len(section_docs)} section documents, {len(table_docs)} table chunks "
      f"(from {sum(b['type'] == 'table' for b in blocks)} tables)")

## Strategy 1: fixed-size

In [ ]:
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=150)
fixed_chunks = fixed_splitter.split_documents(section_docs) + table_docs
print(len(fixed_chunks), "fixed-size chunks")

## Strategy 2: semantic

Embeds every sentence, then starts a new chunk wherever the gap in meaning between neighbouring sentences is in the top 10% (`percentile`, 90). Takes ~30 seconds on CPU; the first run also downloads the model.

On its own it sometimes merges long runs of similar accounting text into one huge chunk (up to ~12,000 characters), which the embedding model would mostly ignore. So any semantic chunk over `CHUNK_SIZE` is re-split at paragraph/sentence boundaries, with no overlap.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
semantic_splitter = SemanticChunker(
    embeddings, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=90
)
raw_semantic = semantic_splitter.split_documents(section_docs)

size_cap = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=0)
semantic_chunks = size_cap.split_documents(raw_semantic) + [
    Document(page_content=d.page_content, metadata=dict(d.metadata)) for d in table_docs
]
print(f"{len(raw_semantic)} semantic text chunks before the size cap, "
      f"{len(semantic_chunks)} chunks after (incl. {len(table_docs)} table chunks)")

## Compare

`all-MiniLM-L6-v2` reads at most **256 tokens** per chunk and ignores the rest when embedding, so the share of chunks over that limit matters: anything past it won't count when searching (the full text still goes to Claude).

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
tokenizer.model_max_length = 1_000_000  # we only count tokens here; silences the "longer than 512" notice
strategies = {"fixed": fixed_chunks, "semantic": semantic_chunks}

for name, chunks in strategies.items():
    for i, c in enumerate(chunks):
        c.metadata.update(strategy=name, chunk_id=f"{name}-{i:04d}")

rows = [
    {"strategy": name, "type": c.metadata["type"], "chars": len(c.page_content),
     "tokens": len(tokenizer.encode(c.page_content, add_special_tokens=False))}
    for name, chunks in strategies.items() for c in chunks
]
stats = pd.DataFrame(rows)

comparison = stats.groupby(["strategy", "type"]).agg(
    chunks=("chars", "size"),
    median_chars=("chars", "median"),
    max_chars=("chars", "max"),
    median_tokens=("tokens", "median"),
    over_256_tokens=("tokens", lambda t: f"{(t > 256).mean():.0%}"),
)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for name in strategies:
    ax.hist(stats.query("strategy == @name and type == 'text'")["tokens"], bins=40, alpha=0.6, label=name)
ax.axvline(256, color="gray", linestyle="--", linewidth=1)
ax.text(260, ax.get_ylim()[1] * 0.9, "model limit (256)", color="gray", fontsize=9)
ax.set_xlabel("tokens per text chunk")
ax.set_ylabel("chunks")
ax.set_title("Text chunk sizes by strategy")
ax.legend()
plt.tight_layout()
plt.show()

## Side by side

The chunk each strategy makes around the same passage in MD&A. Change `PHRASE` or `SECTION` to look at other parts of the filing.

In [ ]:
PHRASE = "net interest margin"
SECTION = "Item 2."

for name, chunks in strategies.items():
    match = next(
        c for c in chunks
        if c.metadata["type"] == "text" and SECTION in c.metadata["section"] and PHRASE in c.page_content.lower()
    )
    print(f"=== {name} · {match.metadata['chunk_id']} · {match.metadata['section']} · {len(match.page_content)} chars ===")
    print(match.page_content)
    print()

## Save

One JSON object per line: `id`, `text`, `metadata` (source, section, type, strategy, and caption for tables).

In [ ]:
for name, chunks in strategies.items():
    path = CHUNKS_DIR / f"{name}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps({"id": c.metadata["chunk_id"], "text": c.page_content, "metadata": c.metadata},
                               ensure_ascii=False) + "\n")
    print("Saved:", path, f"({len(chunks)} chunks, {path.stat().st_size / 1_000:.0f} KB)")

## Verify the saved files

Confirms the whole filing made it into the chunks, using only the files on disk. It doesn't need the cells above or the embedding model, so you can run it on its own in a fresh kernel.

- **Files intact:** every line is valid JSON, the ids are unique and in order, and no chunk is empty.
- **Text covered:** every parsed paragraph is cut into 40-character pieces, and each piece must be found in the text chunks. Spaces and line breaks are ignored, because the splitters change them.
- **Tables covered:** every row of every table must be found in a table chunk.
- **Sections covered:** every Part/Item in the parsed file appears in the chunks.

In [ ]:
import json
import re
from pathlib import Path

PARSED_PATH = Path("../data/parsed/acre-20260630.json")
CHUNKS_DIR = Path("../data/chunks")
PIECE = 40


def squash(text):
    return re.sub(r"\s+", "", text)


def section_of(block):
    return " · ".join(x for x in (block["part"], block["item"]) if x)


blocks = json.loads(PARSED_PATH.read_text(encoding="utf-8"))
parsed_sections = {section_of(b) for b in blocks}

for name in ["fixed", "semantic"]:
    lines = (CHUNKS_DIR / f"{name}.jsonl").read_text(encoding="utf-8").splitlines()
    records = [json.loads(line) for line in lines if line]  # stops with an error if a line was cut off
    texts = [squash(r["text"]) for r in records if r["metadata"]["type"] == "text"]
    tables = [squash(r["text"]) for r in records if r["metadata"]["type"] == "table"]
    joined = "".join(texts)

    ids_ok = [r["id"] for r in records] == [f"{name}-{i:04d}" for i in range(len(records))]
    empty = sum(not r["text"].strip() for r in records)

    pieces = [
        s[i:i + PIECE]
        for b in blocks if b["type"] == "text"
        for s in [squash(b["text"])]
        for i in range(0, len(s), PIECE)
    ]
    missing_text = [p for p in pieces if p not in joined and not any(p in t for t in texts)]

    rows = [squash(line) for b in blocks if b["type"] == "table" for line in b["text"].splitlines()]
    rows = [r for r in rows if r]
    missing_rows = [r for r in rows if not any(r in t for t in tables)]

    missing_sections = parsed_sections - {r["metadata"]["section"] for r in records}

    print(f"=== {name}: {len(records)} chunks ===")
    print(f"  ids unique and in order: {ids_ok}   empty chunks: {empty}")
    print(f"  text pieces found:  {len(pieces) - len(missing_text)} / {len(pieces)}")
    print(f"  table rows found:   {len(rows) - len(missing_rows)} / {len(rows)}")
    print(f"  sections missing:   {sorted(missing_sections) or 'none'}")
    for p in missing_text[:5]:
        print("  missing text:", p)
    for r in missing_rows[:5]:
        print("  missing row:", r[:80])

## See it: parsed text vs. chunks

A visual check for one section at a time, also from the files on disk (no model needed).

Three columns, side by side:

1. **Parsed**: the paragraphs `02_parse` pulled out of the filing, one grey box each.
2. **Fixed** and 3. **Semantic**: the chunks each strategy cut from those same paragraphs, alternating blue and yellow so you can see where one chunk ends and the next begins.

Text highlighted in **red** at the top of a chunk is **overlap**: text repeated from the end of the chunk before it. Only fixed chunks have overlap. Tables are left out here because they're chunked separately.

Run the first cell to list the sections. Then set `SECTION` to any part of a section name and run the second cell.

In [2]:
import json
from pathlib import Path

import pandas as pd

PARSED_PATH = Path("../data/parsed/acre-20260630.json")
CHUNKS_DIR = Path("../data/chunks")


def section_of(block):
    return " · ".join(x for x in (block["part"], block["item"]) if x)


def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]


blocks = json.loads(PARSED_PATH.read_text(encoding="utf-8"))
saved = {name: load_jsonl(CHUNKS_DIR / f"{name}.jsonl") for name in ["fixed", "semantic"]}

sections = pd.DataFrame(
    [{"section": section_of(b), "count": "parsed paragraphs"} for b in blocks if b["type"] == "text"]
    + [{"section": r["metadata"]["section"], "count": f"{name} chunks"}
       for name, records in saved.items() for r in records if r["metadata"]["type"] == "text"]
)
pd.crosstab(sections["section"], sections["count"]).sort_index()

count,fixed chunks,parsed paragraphs,semantic chunks
section,,,
Cover page,12,63,12
Part I · Item 1. Consolidated Financial Statements,150,250,149
Part I · Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations,68,92,70
Part I · Item 3. Quantitative and Qualitative Disclosures About Market Risk,21,25,24
Part I · Item 4. Controls and Procedures,4,4,3
Part II · Item 1. Legal Proceedings,1,1,1
Part II · Item 1A. Risk Factors,1,1,2
Part II · Item 2. Unregistered Sales of Equity Securities and Use of Proceeds,1,3,2
Part II · Item 3. Defaults Upon Senior Securities,1,1,1


The table above shows how many paragraphs and chunks each section has. Start with a small one, such as Controls and Procedures. If `SECTION` matches more than one section (e.g. "Item 4." is in both Part I and Part II), the cell stops and lists them so you can be more specific.

In [3]:
import html

from IPython.display import HTML, display

SECTION = "Controls and Procedures"  # any part of a section name from the table above

CHUNK_COLORS = ["#dbeafe", "#fef3c7"]  # alternate so neighbouring chunks are easy to tell apart
OVERLAP_COLOR = "#fca5a5"


def overlap(prev, cur, min_len=20):
    """Length of the start of `cur` that repeats the end of `prev` (0 if under min_len)."""
    for n in range(min(len(prev), len(cur)), min_len - 1, -1):
        if prev.endswith(cur[:n]):
            return n
    return 0


def box(label, body, color):
    return (f'<div style="background:{color};color:#111;margin:4px 0;padding:6px 8px;border-radius:4px;'
            f'white-space:pre-wrap;font-size:12px;line-height:1.4"><b>{label}</b>\n{body}</div>')


def column(title, boxes):
    return f'<div style="flex:1;min-width:0"><h4 style="margin:4px 0">{title}</h4>{"".join(boxes)}</div>'


matches = sorted({section_of(b) for b in blocks if SECTION in section_of(b)})
assert len(matches) == 1, f"SECTION should match exactly one section, got: {matches}"
section = matches[0]

parsed_boxes = [
    box(f"block {i} · {len(b['text'])} chars", html.escape(b["text"]), "#e5e7eb")
    for i, b in enumerate(blocks)
    if b["type"] == "text" and section_of(b) == section
]

columns = [column(f"Parsed ({len(parsed_boxes)} paragraphs)", parsed_boxes)]
for name, records in saved.items():
    texts = [r for r in records if r["metadata"]["type"] == "text" and r["metadata"]["section"] == section]
    boxes, prev = [], ""
    for k, r in enumerate(texts):
        t = r["text"]
        n = overlap(prev, t)
        body = (f'<span style="background:{OVERLAP_COLOR}">{html.escape(t[:n])}</span>' if n else "") + html.escape(t[n:])
        boxes.append(box(f"{r['id']} · {len(t)} chars" + (f" · {n} overlap" if n else ""), body, CHUNK_COLORS[k % 2]))
        prev = t
    columns.append(column(f"{name.capitalize()} ({len(texts)} chunks)", boxes))

display(HTML(
    f"<h3>{html.escape(section)}</h3>"
    f'<div style="display:flex;gap:10px;max-height:900px;overflow:auto">{"".join(columns)}</div>'
))